# Lab-2: Can You Find the Complaint?
### Search baselines, and how to tell whether your search is any good
**AML-3203 · Lab 2 · the first lab in the Social Listening Copilot series · ≈ 90 minutes**

Over this series of labs you will build a system that reads a brand's social media mentions and answers questions about them — with citations, and without making things up. This is the first layer: **finding the right posts**.

Everything downstream depends on retrieval. An answering system that retrieves the wrong posts will give you a confident, well-written, wrong answer. So before any of the clever machinery, we need two things: a way to search, and **a way to know whether the search is working.**

## The situation

Aurora makes wireless earbuds. You have **1,950 posts** about them from five platforms over six months. The head of product asks:

> *"Customer complaints feel like they're getting worse. What are people actually unhappy about?"*

## What you'll do

| Part | |
|---|---|
| 1 | Meet the corpus |
| 2 | Search it the obvious way, and feel good about the results |
| 3 | Build an evaluation harness — the thing most people skip |
| 4 | Measure the obvious way. Discover its ceiling. |
| 5 | Try TF-IDF and measure again |
| 6 | Find out why the average score is lying to you |

**No API keys, no downloads.** Everything here is pandas and scikit-learn.

> These labs are not graded. The assert cells exist so you know your code is right before you build the next thing on top of it.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 95)

posts = pd.read_csv("social_posts.csv")
print(posts.shape)
posts.head(5)

## Part 1 — Meet the corpus

This is text data, which behaves differently from the tables you've worked with. Before searching it, get a feel for it.

Look at: how many posts, how many authors, the platform mix, the `post_type` mix, and the distribution of post lengths. Then **read twenty random posts.** Actually read them — you cannot design a search for text you have never looked at.

In [ ]:
# TODO: size, authors, date range, platform mix, post_type mix, length distribution

In [ ]:
for t in posts.text.sample(20, random_state=7):
    print("-", t)

**Notice the vocabulary.** People describing the same problem use completely different words. Hold that thought — it is the whole lab.

## Part 2 — Search the obvious way

Someone asks about battery complaints. The obvious move is to search for the word.

Do that: find every post containing `"battery"` (case-insensitive). How many are there? Read a few. **Does this look like a good answer to the head of product's question?**

Then do the same for `"fit"` to find comfort complaints.

In [ ]:
# TODO: find posts containing 'battery', and posts containing 'fit'.
#       How many of each? Read a few. Are you satisfied?

A few dozen posts about battery problems out of 1,950. That is a plausible-looking number, and if you stopped here you would report it with a straight face.

**How would you know if it were wrong?**

That question is the real content of this lab. You cannot answer it by looking at the results you found — you would be checking your answer against itself. You need to know what you *should* have found.

## Part 3 — Build the evaluation harness

`eval_questions.csv` contains ten questions. For each one, somebody has gone through the corpus and labelled **every post that genuinely answers it**. That labelling is the ground truth.

This is the part everyone skips, and it is the part that makes everything afterwards meaningful. Without it you are tuning a search engine by vibes.

**3a.** Load it. For each question print the number of relevant posts.

**3b.** Write two functions:

- `precision_at_k(retrieved_ids, relevant_ids, k)` — of the top *k* results, what fraction are relevant? *Did it waste my time?*
- `recall_at_k(retrieved_ids, relevant_ids, k)` — of all the relevant posts, what fraction did the top *k* contain? *Did it miss things?*

**3c.** One question has **zero** relevant posts. Find it, and work out why that is there on purpose.

In [ ]:
# TODO 3a: load eval_questions.csv and build a dict: question_id -> set of relevant post_ids
#          (the ids are pipe-separated in one column)

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k=10):
    # TODO: of the top k retrieved, what fraction are relevant?
    ...


def recall_at_k(retrieved_ids, relevant_ids, k=10):
    # TODO: of all relevant posts, what fraction appear in the top k?
    #       Think about what this should return when relevant_ids is empty.
    ...

In [ ]:
# ---- self-check ----
assert precision_at_k(["a", "b", "c", "d"], {"a", "c"}, k=4) == 0.5
assert precision_at_k(["a", "b", "c", "d"], {"a", "c"}, k=2) == 0.5
assert recall_at_k(["a", "b"], {"a", "c", "d", "e"}, k=2) == 0.25
assert np.isnan(recall_at_k(["a"], set(), k=10))
print("metrics OK")

**3c — the question with nothing relevant.** Find it in code, read the question, and check the corpus yourself. Then write one sentence: why would someone put a question with **no** answer into an eval set on purpose?

*Hint: think about what a system that writes answers in plain English would do with this question.*

In [ ]:
# TODO 3c: which question has zero relevant posts? Does the corpus mention it at all?


**Your answer (3c):** why is this question in the eval set?


### A trap worth understanding before you measure anything

Look at `n_relevant` for Q01: **271 posts**. Now ask what `recall@10` could possibly be, at best.

Ten results, 271 relevant posts. Even a perfect search engine scores `10/271 = 3.7%`.

**`recall@k` can never exceed `k / n_relevant`.** When the relevant set is much bigger than *k*, recall measures your choice of *k*, not your search. For this corpus, **precision@10 is the honest headline metric** — *of the ten things you showed me, how many were useful?* — and recall is worth reporting at a larger *k*, with the ceiling in mind.

Compute that ceiling for each question now, so you don't misread your own results later.

In [ ]:
# TODO: for each question, the maximum achievable recall@10 and recall@50
#       (skip Q08 - it has no relevant posts, so there is nothing to divide by)

## Part 4 — Measure the obvious approach

Now put keyword search on the scoreboard.

Write `keyword_search(query, k)` that scores each post by how many of the query's words it contains, and returns the top *k* post IDs. Run it on all ten questions and report precision@10.

Keep your results in a DataFrame called **`ev`**, one row per question, with columns **`question_id`** and **`keyword_p@10`**. You will add to this table for the rest of the series.

Then the question that matters: **for Q01, what is the best recall a keyword search for "battery" could ever achieve on this corpus?** You know how many posts contain the word, and the eval set tells you how many are genuinely about the problem.

In [ ]:
# TODO: write keyword_search(query, k) - score posts by how many query terms they contain,
#       return the top k post_ids. Then score all ten questions with precision@10.
#       Store the scores in a DataFrame `ev` with columns 'question_id' and 'keyword_p@10'.
#       (A small stopword list will help - 'what', 'are', 'the' match everything.)

In [ ]:
# TODO: how many posts contain the literal word 'battery', how many are actually
#       about battery life, and what does that imply about the best recall a
#       keyword search could ever reach? Repeat for 'fit'.

**Sit with that number.** A keyword search for "battery" has a hard ceiling of about **17%** on this corpus — and for the fit complaints, about **24%**. Not because the search is badly written, but because **more than four in five people complaining about battery life never use the word.**

They write *"these die by lunchtime"*, *"won't hold a charge"*, *"dead before my commute home"*, *"I charge them overnight and they're flat by 11am"*.

No amount of tuning fixes this. The method has a ceiling, and the ceiling is below what the business needs. Finding that out *before* you report a number is the difference between analysis and guessing.

## Part 5 — TF-IDF: a real retrieval baseline

Keyword counting treats every word as equally important, so a word that appears in hundreds of posts counts as much as "battery". **TF-IDF** weights each term by how rare it is across the corpus, so distinctive words dominate the match.

Use `TfidfVectorizer` to build a matrix over the posts, then score a query by cosine similarity against every post. Run the same ten questions. Report precision@10 next to the keyword column.

**Before you run it, predict:** will TF-IDF beat your keyword search? On which questions?

In [ ]:
# TODO: build a TfidfVectorizer over posts.text and call the matrix `X`.
#       Write tfidf_search(query, k) using cosine similarity, score all ten
#       questions, and add the scores to `ev` as a column 'tfidf_p@10'.

In [ ]:
# ---- self-check ----
assert X.shape[0] == len(posts)
assert len(tfidf_search("charging case", 10)) == 10
grounded = ev[ev.question_id != "Q08"]   # Q08 has no relevant posts, so its precision is always 0
print(f"mean keyword precision@10 across the grounded questions: {grounded['keyword_p@10'].mean():.1%}")
print(f"mean TF-IDF  precision@10 across the grounded questions: {grounded['tfidf_p@10'].mean():.1%}")

**About 70%.** If you were reporting to a manager, "our search is 70% precise" sounds like a system that mostly works.

Two surprises. First, it is **no better on average than counting keywords** — your keyword mean is probably within a few points of it, and may even be higher. Weighting words more cleverly cannot help when the right posts do not share the words at all. Both methods hit the same wall.

Second, the 70% itself is misleading. Look at the column, not the average.

## Part 6 — Why the average is lying

**6a.** Sort the questions by `tfidf_p@10` (leave out Q08, which has no relevant posts). What does the distribution look like? Is 70% a typical score for any question?

**6b.** Take the worst question (again, not Q08). Print the ten posts TF-IDF actually returned, and ten posts from the relevant set it was supposed to find. Put them side by side and explain the failure in one sentence.

**6c.** Which kinds of question does TF-IDF handle perfectly, and which does it fail completely? What distinguishes the two groups?

In [ ]:
# TODO 6a: sort by tfidf_p@10 and look at the shape of the distribution

In [ ]:
# TODO 6b: for the worst question, print what TF-IDF returned next to what it
#          should have found. The explanation should be obvious once you see them.

**Your answers:**

- 6a: what does the distribution of scores look like?
- 6b: why did the worst question fail? One sentence.
- 6c: what separates the questions it aces from the ones it fails?

## Where this leaves us

You now have three things, and the third is the valuable one:

1. A keyword baseline with a measured ceiling of ~17% recall on the question that matters (battery life).
2. A TF-IDF baseline that is perfect on literal questions, scores zero on the most paraphrased ones (Q01, Q02), and on average is no better than keyword counting.
3. **A reusable evaluation harness.** Ten questions, labelled answers, two metrics, and a scoreboard.

Every lab in this series re-runs these same ten questions. Nothing gets adopted because it is fashionable — it gets adopted because the number moves.

### About 3c — the question with no answer

`Q08` asks *"What are people saying about the Aurora Watch?"* Aurora does not make a watch, and no post mentions one.

It is in the eval set because in Lab-6 you will build a system that writes answers in natural language, and that system will cheerfully produce a detailed paragraph about the Aurora Watch unless you make it refuse. **A question with no answer is the only way to test whether a system knows what it doesn't know.** Keep it.

### The problem to solve next

Matching *words* cannot find *meaning*. "Dies by lunchtime" and "poor battery life" share nothing lexically and everything semantically.

In **Lab-3** you will represent posts as vectors that encode meaning rather than vocabulary, and re-run this exact scoreboard. Keep your `ev` table — you will add a column to it.

One thing to watch for next time, so you aren't taken in twice: there are two questions in this set where TF-IDF scores 100% and the new method may well do **worse**. See if you can predict which, and why.

In [ ]:
ev.to_csv("eval_scoreboard.csv", index=False)
print("saved eval_scoreboard.csv - bring this to Lab-3")
print()
print(ev[["question_id", "keyword_p@10", "tfidf_p@10"]].to_string(index=False))